In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# find the shared folder
possible = [
    '/content/drive/MyDrive/HeyCareLog_Dataset',
    '/content/drive/Shareddrives/HeyCareLog_Dataset',
]

BASE = None
for p in possible:
    if os.path.exists(p):
        BASE = p
        break

if BASE is None:
    for root, dirs, files in os.walk('/content/drive'):
        for d in dirs:
            if 'HeyCareLog' in d:
                BASE = os.path.join(root, d)
                break
        if BASE:
            break

print(f'BASE = {BASE}')
print('Contents:', os.listdir(BASE))

os.makedirs(f'{BASE}/models/whisper_finetuned', exist_ok=True)
os.makedirs(f'{BASE}/results', exist_ok=True)

print('Drive connected and folders ready!')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE = /content/drive/MyDrive/HeyCareLog_Dataset
Contents: ['audio', 'labels', 'podcastfillers', 'models', 'results', 'notebooks']
Drive connected and folders ready!


In [ ]:
import shutil, os

broken_path = f'{BASE}/models/whisper_finetuned'

if os.path.exists(broken_path):
    shutil.rmtree(broken_path)
    print('Deleted broken checkpoints')

os.makedirs(broken_path, exist_ok=True)
print('Ready to re-train!')

Deleted broken checkpoints
Ready to re-train!


Install Libraries

In [ ]:
!pip install -q transformers
!pip install -q datasets
!pip install -q accelerate
!pip install -q evaluate
!pip install -q jiwer
!pip install -q soundfile
!pip install -q librosa

import torch
print('Libraries installed!')
print(f'GPU available: {torch.cuda.is_available()}')
print(f'GPU name:      {torch.cuda.get_device_name(0)}')

Libraries installed!
GPU available: True
GPU name:      NVIDIA A100-SXM4-40GB


 Load Data

In [ ]:
import pandas as pd, os

train_df = pd.read_csv(f'{BASE}/labels/train_audio.csv')
test_df  = pd.read_csv(f'{BASE}/labels/test_audio.csv')

proc_dir = f'{BASE}/audio/processed'

train_df = train_df[train_df['w'].apply(
    lambda x: os.path.exists(f'{proc_dir}/{x}.wav')
)].reset_index(drop=True)

test_df = test_df[test_df['w'].apply(
    lambda x: os.path.exists(f'{proc_dir}/{x}.wav')
)].reset_index(drop=True)

print(f'Training recordings: {len(train_df)}')
print(f'Test recordings:     {len(test_df)}')
print()
print('Sample:')
print(f'  ID:   {train_df["w"].iloc[0]}')
print(f'  Text: {str(train_df["speech_to_text_output"].iloc[0])[:120]}')

Training recordings: 177
Test recordings:     23

Sample:
  ID:   V0340
  Text: Today is March 9 2026 this is for patient P028 in the male branch Morning care was done at 7:41 AM He had uh full bath w


Prepare Audio Features

In [ ]:
import librosa
from transformers import WhisperProcessor

MODEL_NAME = 'openai/whisper-small'
processor  = WhisperProcessor.from_pretrained(MODEL_NAME)

proc_dir = f'{BASE}/audio/processed'

def prepare_example(row):
    audio, _ = librosa.load(
        f'{proc_dir}/{row["w"]}.wav', sr=16000
    )
    features = processor.feature_extractor(
        audio,
        sampling_rate=16000,
        return_tensors='pt'
    ).input_features[0]

    labels = processor.tokenizer(
        str(row['speech_to_text_output']),
        return_tensors='pt'
    ).input_ids[0]

    return {'input_features': features, 'labels': labels}

print('Preparing training data...')
train_data = []
for _, row in train_df.iterrows():
    try:
        train_data.append(prepare_example(row))
    except Exception as e:
        print(f'Skipping {row["w"]}: {e}')

print('Preparing test data...')
test_data = []
for _, row in test_df.iterrows():
    try:
        test_data.append(prepare_example(row))
    except Exception as e:
        print(f'Skipping {row["w"]}: {e}')

print(f'Training examples: {len(train_data)}')
print(f'Test examples:     {len(test_data)}')

Preparing training data...
Preparing test data...
Training examples: 177
Test examples:     23


 Data Collator

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List

@dataclass
class WhisperDataCollator:
    processor: Any

    def __call__(
        self, features: List[Dict[str, Any]]
    ) -> Dict[str, torch.Tensor]:

        input_features = [
            {'input_features': f['input_features']}
            for f in features
        ]
        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors='pt'
        )

        label_features = [
            {'input_ids': f['labels']} for f in features
        ]
        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors='pt'
        )

        labels = labels_batch['input_ids'].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        if (labels[:, 0] ==
                self.processor.tokenizer.bos_token_id).all():
            labels = labels[:, 1:]

        batch['labels'] = labels
        return batch

data_collator = WhisperDataCollator(processor=processor)
print('Data collator ready!')

Data collator ready!


WER Metric

In [ ]:
import evaluate

wer_metric = evaluate.load('wer')

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = (
        processor.tokenizer.pad_token_id
    )

    pred_str = processor.tokenizer.batch_decode(
        pred_ids, skip_special_tokens=True
    )
    label_str = processor.tokenizer.batch_decode(
        label_ids, skip_special_tokens=True
    )

    wer = wer_metric.compute(
        predictions=pred_str,
        references=label_str
    )
    return {'wer': round(wer * 100, 2)}

print('WER metric ready!')
print('Current pre-trained WER: 27.22%')
print('Target WER: below 20%')

WER metric ready!
Current pre-trained WER: 27.22%
Target WER: below 20%


Load Model and Configure Training

In [ ]:
from transformers import (
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

print('Loading Whisper-small...')
model = WhisperForConditionalGeneration.from_pretrained(
    'openai/whisper-small'
)

training_args = Seq2SeqTrainingArguments(
    output_dir=f'{BASE}/models/whisper_finetuned',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='wer',
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=225,
    fp16=True,
    logging_steps=10,
    report_to=['none'],
)

print('Model ready!')
print(f'Training on {len(train_data)} recordings')
print(f'Evaluating on {len(test_data)} recordings')
print()
print('Settings:')
print('  Epochs:     3')
print('  Batch size: 4')
print('  LR:         1e-5')
print('  Warmup:     100')

Loading Whisper-small...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Model ready!
Training on 177 recordings
Evaluating on 23 recordings

Settings:
  Epochs:     3
  Batch size: 4
  LR:         1e-5
  Warmup:     100


Fine-Tune

In [ ]:
from torch.utils.data import Dataset as TorchDataset

class AudioDataset(TorchDataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

train_dataset = AudioDataset(train_data)
test_dataset  = AudioDataset(test_data)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

print('='*50)
print('WHISPER-SMALL FINE-TUNING')
print('='*50)
print('Pre-trained WER: 27.22%')
print('Target WER:      below 20%')
print('Expected time:   15-20 min on A100')
print()
print('Watch eval_wer decrease each epoch.')
print('Keep this tab open.')
print()

trainer.train()

print()
print('Fine-tuning complete!')
print(f'Saved to: {BASE}/models/whisper_finetuned')

WHISPER-SMALL FINE-TUNING
Pre-trained WER: 27.22%
Target WER:      below 20%
Expected time:   15-20 min on A100

Watch eval_wer decrease each epoch.
Keep this tab open.



Epoch,Training Loss,Validation Loss,Wer
1,6.149982,2.508060,74.360000
2,3.365680,0.966042,93.870000
3,1.170080,0.378355,83.630000


Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].



Fine-tuning complete!
Saved to: /content/drive/MyDrive/HeyCareLog_Dataset/models/whisper_finetuned


Check Final WER

In [ ]:
import os, jiwer, torch, librosa, gc
from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor
)

proc_dir = f'{BASE}/audio/processed'

ft_base = f'{BASE}/models/whisper_finetuned'
checkpoints = sorted([
    d for d in os.listdir(ft_base)
    if d.startswith('checkpoint')
])
BEST = f'{ft_base}/{checkpoints[-1]}'
print(f'Best checkpoint: {BEST}')

print('Loading fine-tuned model...')
ft_processor = WhisperProcessor.from_pretrained(BEST)
ft_model     = WhisperForConditionalGeneration.from_pretrained(BEST)
ft_model.eval()

preds = []
refs  = []

print('Evaluating on 23 test recordings...')
for _, row in test_df.iterrows():
    path = f'{proc_dir}/{row["w"]}.wav'
    if not os.path.exists(path):
        continue

    audio, _ = librosa.load(path, sr=16000)
    inputs   = ft_processor.feature_extractor(
        audio, sampling_rate=16000, return_tensors='pt'
    ).input_features

    with torch.no_grad():
        ids = ft_model.generate(
            inputs,
            language='english',
            task='transcribe'
        )

    pred = ft_processor.tokenizer.decode(
        ids[0], skip_special_tokens=True
    )
    preds.append(pred.lower().strip())
    refs.append(
        str(row['speech_to_text_output']).lower().strip()
    )
    print(f'  {row["w"]}: {pred[:60]}')

wer_ft         = round(jiwer.wer(refs, preds) * 100, 2)
wer_pretrained = 27.22
improvement    = round(wer_pretrained - wer_ft, 2)

print()
print('='*50)
print('WHISPER FINE-TUNING RESULTS')
print('='*50)
print(f'Whisper-tiny  (pre-trained): 37.68%')
print(f'Whisper-base  (pre-trained): 31.69%')
print(f'Whisper-small (pre-trained): {wer_pretrained}%')
print(f'Whisper-small (fine-tuned):  {wer_ft}%  <- BEST')
print(f'Improvement:                 {improvement}%')
print()
if wer_ft < 20:
    print('TARGET MET: WER below 20%')
else:
    print(f'WER = {wer_ft}%')
print('='*50)

import pandas as pd
pd.DataFrame([
    {'Model': 'Whisper-tiny (pre-trained)',  'WER_%': 37.68},
    {'Model': 'Whisper-base (pre-trained)',  'WER_%': 31.69},
    {'Model': 'Whisper-small (pre-trained)', 'WER_%': wer_pretrained},
    {'Model': 'Whisper-small (fine-tuned)',  'WER_%': wer_ft},
]).to_csv(
    f'{BASE}/results/whisper_final_results.csv', index=False
)
print('Results saved!')

del ft_model, ft_processor
gc.collect()
torch.cuda.empty_cache()

Best checkpoint: /content/drive/MyDrive/HeyCareLog_Dataset/models/whisper_finetuned/checkpoint-69
Loading fine-tuned model...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Evaluating on 23 test recordings...
  V0512: 
  V0273: 
  V0341: 
  V0333: 
  V0428: 
  V0413: 
  V0494: 
  V0516: 
  V0320: 
  V0268: 
  V0037: 
  V0281: 
  V0323: 
  V0280: 
  V0342: 
  V0414: 
  V0080: 
  V0377: 
  V0043: 
  V0493: 
  V0347: 
  V0084: 
  V0090: 

WHISPER FINE-TUNING RESULTS
Whisper-tiny  (pre-trained): 37.68%
Whisper-base  (pre-trained): 31.69%
Whisper-small (pre-trained): 27.22%
Whisper-small (fine-tuned):  100.0%  <- BEST
Improvement:                 -72.78%

WER = 100.0%
Results saved!
